# LeetCode 2887 — Fill Missing Data bằng Python (pandas)

Link đề: https://leetcode.com/problems/fill-missing-data/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def fillMissingValues(products: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bảng `products` có cột `name`, `quantity`, `price`; **thay mọi giá trị thiếu ở cột `quantity` bằng `0`**.
2. Hiểu "giá trị thiếu" (`NaN`) và vì sao cột có `NaN` thường là **`float64`** (số thực), kể cả khi các giá trị còn lại là số nguyên.
3. Đi từ cách **vòng lặp tự thay từng ô** → **`fillna(0)` trên cột** → **`DataFrame.fillna({"quantity": 0})` theo từ điển** → **`where` / `mask`** → **gán bằng `loc` và mặt nạ `isna()`**.
4. Biết **kiểu dữ liệu không tự đổi sau khi lấp** (`float64` vẫn là `float64`: `0.0`, không thành `0`), và khi nào cần `astype(int)`.
5. Tránh các lỗi kinh điển: `fillna(0)` trên **cả bảng** làm hỏng cột khác, quên gán lại, `inplace=True` trên cột (chained assignment), `fillna(method=...)` đã bị gỡ ở pandas 3.

## 2. Đề bài & ví dụ

Có một bảng `products` với các cột `name`, `quantity`, `price`. Một số sản phẩm **chưa có số lượng** (`quantity` bị thiếu).

Hãy viết hàm **điền `0` vào mọi ô thiếu của cột `quantity`** và trả về bảng đã sửa.

**Ví dụ**

- Input: DataFrame `products`

| name | quantity | price |
|---|---|---|
| Wristwatch | None | 135 |
| WirelessEarbuds | None | 821 |
| GolfClubs | 779 | 9319 |
| Printer | 849 | 3051 |

- Output:

| name | quantity | price |
|---|---|---|
| Wristwatch | 0 | 135 |
| WirelessEarbuds | 0 | 821 |
| GolfClubs | 779 | 9319 |
| Printer | 849 | 3051 |

- Giải thích: hai sản phẩm đầu thiếu số lượng nên được điền `0`; hai sản phẩm sau giữ nguyên. Ở pandas, cột `quantity` có `NaN` nên là `float64`; sau `fillna(0)` các giá trị là `0.0`, `779.0`, ... (hiển thị `0.0`). Việc đề muốn `0.0` hay `0` nên đối chiếu lại với đề gốc; trình chấm của LeetCode thường so giá trị.

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

products = pd.DataFrame({
    "name": ["Wristwatch", "WirelessEarbuds", "GolfClubs", "Printer"],
    "quantity": [None, None, 779, 849],
    "price": [135, 821, 9319, 3051],
})
print(products)
print()
print(products.dtypes)
print("pandas", pd.__version__)

              name  quantity  price
0       Wristwatch       NaN    135
1  WirelessEarbuds       NaN    821
2        GolfClubs     779.0   9319
3          Printer     849.0   3051

name            str
quantity    float64
price         int64
dtype: object
pandas 3.0.5


## 3. Kiến thức nền

### 3.1. Vì sao cột `quantity` là `float64`?

`NaN` là một **số thực đặc biệt**, nên NumPy không thể đặt `NaN` vào một cột số nguyên `int64`. Vì vậy khi trong cột số có ô thiếu, pandas chuyển **cả cột** sang `float64`: `779` thành `779.0`. Đây là lý do bạn thấy dấu `.0` trong cột `quantity` của bảng.

In [2]:
print(products["quantity"].tolist())            # [nan, nan, 779.0, 849.0]
print(products["quantity"].dtype)               # float64
print(pd.Series([779, 849]).dtype)              # int64: không có NaN thì vẫn là số nguyên

[nan, nan, 779.0, 849.0]
float64
int64


### 3.2. Tìm ô thiếu: `isna()` và `notna()`

`s.isna()` trả Series `True/False`: `True` ở ô **thiếu**. `s.notna()` ngược lại. Có thể **đếm** ô thiếu bằng `s.isna().sum()` (vì `True` tính là 1). **Không** dùng `== NaN` hay `== None` (so sánh luôn cho `False`).

In [3]:
q = products["quantity"]
print(q.isna().tolist())          # [True, True, False, False]
print(q.notna().tolist())         # [False, False, True, True]
print("số ô thiếu:", q.isna().sum())
print(products.isna().sum().to_dict())    # đếm thiếu của MỌI cột

[True, True, False, False]
[False, False, True, True]
số ô thiếu: 2
{'name': 0, 'quantity': 2, 'price': 0}


### 3.3. `Series.fillna(giá_trị)`

`s.fillna(0)` trả về **Series mới** trong đó mọi ô thiếu được thay bằng `0`. Các ô có giá trị giữ nguyên. Series cũ **không đổi**; muốn lưu kết quả phải gán lại (`df["quantity"] = df["quantity"].fillna(0)`).

In [4]:
filled = q.fillna(0)
print(filled.tolist(), "| dtype:", filled.dtype)    # [0.0, 0.0, 779.0, 849.0] float64
print(q.tolist())                                   # Series gốc vẫn còn nan

[0.0, 0.0, 779.0, 849.0] | dtype: float64
[nan, nan, 779.0, 849.0]


### 3.4. `DataFrame.fillna(từ_điển)`

Với **cả bảng**, `df.fillna({"quantity": 0})` chỉ lấp **các cột nêu trong từ điển**, mỗi cột một giá trị riêng. Trả **bảng mới**. Ngược lại, `df.fillna(0)` (không có từ điển) sẽ cố điền `0` vào **mọi cột**, kể cả cột chữ; ta sẽ thấy hậu quả ở mục 13.

In [5]:
out = products.fillna({"quantity": 0})
print(out)
print("bảng gốc còn thiếu:", products["quantity"].isna().sum())

              name  quantity  price
0       Wristwatch       0.0    135
1  WirelessEarbuds       0.0    821
2        GolfClubs     779.0   9319
3          Printer     849.0   3051
bảng gốc còn thiếu: 2


### 3.5. `where` và `mask`

- `s.where(điều_kiện, thay_thế)`: **giữ** giá trị nơi điều kiện `True`, **thay** bằng `thay_thế` nơi `False`.
- `s.mask(điều_kiện, thay_thế)`: ngược lại, **thay** nơi điều kiện `True`.

Với bài này: giữ nơi `notna()` hoặc thay nơi `isna()`: hai cách cho cùng kết quả.

In [6]:
print(q.where(q.notna(), 0).tolist())     # giữ ô có giá trị, ô thiếu -> 0
print(q.mask(q.isna(), 0).tolist())       # ô thiếu -> 0, còn lại giữ

[0.0, 0.0, 779.0, 849.0]
[0.0, 0.0, 779.0, 849.0]


### 3.6. Gán có điều kiện bằng `loc`

`df.loc[mặt_nạ, "cột"] = giá_trị` **gán** giá trị cho các ô của cột nơi mặt nạ là `True`. Đây là **phép gán tại chỗ** nên ta làm trên bản sao `copy()` để giữ dữ liệu mẫu.

In [7]:
tmp = products.copy()
mask = tmp["quantity"].isna()
tmp.loc[mask, "quantity"] = 0          # chỉ ô thiếu của cột quantity bị gán 0
print(tmp)
print("bảng gốc không đổi:", products["quantity"].isna().sum(), "ô thiếu")

              name  quantity  price
0       Wristwatch       0.0    135
1  WirelessEarbuds       0.0    821
2        GolfClubs     779.0   9319
3          Printer     849.0   3051
bảng gốc không đổi: 2 ô thiếu


### 3.7. Kiểu dữ liệu không tự đổi sau khi lấp

Sau `fillna(0)`, cột vẫn là `float64` (`0.0`), chứ **không** tự trở lại `int64`. Nếu cần số nguyên thì thêm `astype(int)` (an toàn vì đã hết `NaN`). Với đề này ta giữ nguyên `float64`; giá trị `0.0` và `0` bằng nhau khi so sánh (`0.0 == 0` là `True`).

In [8]:
filled = products["quantity"].fillna(0)
print(filled.dtype, filled.tolist())
as_int = filled.astype(int)
print(as_int.dtype, as_int.tolist())
print(0.0 == 0)                                  # True

float64 [0.0, 0.0, 779.0, 849.0]
int64 [0, 0, 779, 849]
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng bảng tồn kho trên giấy có vài ô **bỏ trống** ở cột "số lượng". Ta **điền số 0 vào mọi ô trống** của cột đó, ô đã có số thì để nguyên.

| nhãn | name | quantity cũ | ô trống? | quantity mới |
|---|---|---|---|---|
| 0 | Wristwatch | (trống) | có | **0** |
| 1 | WirelessEarbuds | (trống) | có | **0** |
| 2 | GolfClubs | 779 | không | 779 |
| 3 | Printer | 849 | không | 849 |

Thuật toán: **với mỗi ô của cột, nếu thiếu thì thay bằng 0, nếu không thì giữ nguyên, rồi ghi lại vào cột**. Các cột `name`, `price` không động tới.

## 5. Cách 1 — Brute Force / baseline: vòng lặp thuần Python

**Ý tưởng.** Duyệt từng số lượng; nếu `pd.isna(q)` thì thêm `0` vào danh sách mới, ngược lại thêm chính `q`. Cuối cùng gán danh sách (dưới dạng Series `float64`) vào cột.

In [9]:
def fill_loop(products: pd.DataFrame) -> pd.DataFrame:
    df = products.copy()                                  # không đụng bảng gốc
    new_quantity = []
    for q in df["quantity"]:                              # duyệt từng ô của cột
        if pd.isna(q):                                    # ô thiếu
            new_quantity.append(0)
        else:
            new_quantity.append(q)
    df["quantity"] = pd.Series(new_quantity, index=df.index, dtype="float64")
    return df


print(fill_loop(products))

              name  quantity  price
0       Wristwatch       0.0    135
1  WirelessEarbuds       0.0    821
2        GolfClubs     779.0   9319
3          Printer     849.0   3051


In [10]:
res = fill_loop(products)
print("quantity mới:", res["quantity"].tolist())           # [0.0, 0.0, 779.0, 849.0]
print("còn ô thiếu :", res["quantity"].isna().sum())       # 0

quantity mới: [0.0, 0.0, 779.0, 849.0]
còn ô thiếu : 0


**Độ phức tạp (Cách 1).** Gọi `m` là số hàng. Vòng lặp `m` lần, mỗi lần `O(1)`: **Time `O(m)`**; danh sách mới `m` phần tử: **Space `O(m)`**.

**Nút thắt.** Vòng lặp Python chậm với bảng lớn và dài dòng. pandas có `fillna` làm đúng việc này bằng mã C.

## 6. Ý tưởng thuật toán / bước đột phá

pandas gói đúng thao tác ta vừa làm bằng tay:

`products["quantity"] = products["quantity"].fillna(0)`

- `fillna(0)`: thay mọi ô thiếu bằng `0`;
- gán đè lại vào cột `quantity`;
- cột `name`, `price` không bị động tới.

Các cách khác: **`DataFrame.fillna({"quantity": 0})`** (Cách 3), **`where` / `mask`** (Cách 4), **`loc` + mặt nạ** (Cách 5). Biết nhiều cách giúp bạn xử lý biến thể (lấp bằng trung bình, bằng giá trị phía trước, bằng giá trị khác nhau theo cột).

## 7. Các cách tiếp theo

### Cách 2 — `fillna(0)` trên cột rồi gán lại (cách chuẩn)

**Giải thích.** `products["quantity"].fillna(0)` tạo Series mới; ta gán lại vào cột. Sửa luôn `products` đầu vào rồi trả về nó; LeetCode chấp nhận.

In [11]:
def fill_fillna(products: pd.DataFrame) -> pd.DataFrame:
    df = products.copy()
    df["quantity"] = df["quantity"].fillna(0)
    return df


print(fill_fillna(products))

              name  quantity  price
0       Wristwatch       0.0    135
1  WirelessEarbuds       0.0    821
2        GolfClubs     779.0   9319
3          Printer     849.0   3051


In [12]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def fillMissingValues(products: pd.DataFrame) -> pd.DataFrame:
    products["quantity"] = products["quantity"].fillna(0)
    return products

**Độ phức tạp Cách 2:** **Time `O(m)`** (quét cột một lần bằng mã C), **Space `O(m)`** (cột mới).

### Cách 3 — `DataFrame.fillna({"quantity": 0})`

**Giải thích.** Gọi `fillna` trên **cả bảng** với từ điển `{tên_cột: giá_trị}`: trả **bảng mới**, không sửa đầu vào, và dễ mở rộng (`{"quantity": 0, "price": 1}`).

In [13]:
def fill_dict(products: pd.DataFrame) -> pd.DataFrame:
    return products.fillna({"quantity": 0})


before = products.copy()
print(fill_dict(products))
pd.testing.assert_frame_equal(products, before)
print("đầu vào không đổi")

              name  quantity  price
0       Wristwatch       0.0    135
1  WirelessEarbuds       0.0    821
2        GolfClubs     779.0   9319
3          Printer     849.0   3051
đầu vào không đổi


In [14]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def fillMissingValues(products: pd.DataFrame) -> pd.DataFrame:
    return products.fillna({"quantity": 0})

**Độ phức tạp Cách 3:** **Time `O(m)`**, **Space `O(m)` đến `O(m·c)`** (bảng mới; cột không đổi có thể dùng chung bộ nhớ nhờ copy-on-write).

### Cách 4 — `where` / `mask`

**Giải thích.** Tách thành "điều kiện" và "giá trị thay thế": `where(notna, 0)` giữ chỗ có giá trị; `mask(isna, 0)` thay chỗ thiếu. Hữu ích khi điều kiện thay thế **không chỉ là "thiếu"** (ví dụ thay luôn số âm).

In [15]:
def fill_where(products: pd.DataFrame) -> pd.DataFrame:
    q = products["quantity"]
    return products.assign(quantity=q.where(q.notna(), 0))


def fill_mask(products: pd.DataFrame) -> pd.DataFrame:
    q = products["quantity"]
    return products.assign(quantity=q.mask(q.isna(), 0))


print(fill_where(products)["quantity"].tolist())
print(fill_mask(products)["quantity"].tolist())

[0.0, 0.0, 779.0, 849.0]
[0.0, 0.0, 779.0, 849.0]


**Độ phức tạp Cách 4:** **Time `O(m)`**, **Space `O(m)`**.

### Cách 5 — `loc` với mặt nạ `isna()` (gán có điều kiện)

**Giải thích.** `df.loc[df["quantity"].isna(), "quantity"] = 0` chọn đúng các ô thiếu của cột rồi gán `0` tại chỗ. Ưu điểm: tổng quát cho "gán giá trị cho những ô thỏa điều kiện" (không chỉ ô thiếu).

In [16]:
def fill_loc(products: pd.DataFrame) -> pd.DataFrame:
    df = products.copy()
    df.loc[df["quantity"].isna(), "quantity"] = 0
    return df


print(fill_loc(products))

              name  quantity  price
0       Wristwatch       0.0    135
1  WirelessEarbuds       0.0    821
2        GolfClubs     779.0   9319
3          Printer     849.0   3051


In [17]:
# Code nộp LeetCode cho Cách 5
import pandas as pd

def fillMissingValues(products: pd.DataFrame) -> pd.DataFrame:
    products.loc[products["quantity"].isna(), "quantity"] = 0
    return products

**Độ phức tạp Cách 5:** **Time `O(m)`**, **Space `O(m)`** (mặt nạ bool).

> Cách 2 và Cách 3 ngắn nhất, dễ đọc nhất. Cách 4 và Cách 5 đáng nhớ vì dùng lại được cho mọi bài "thay giá trị theo điều kiện".

## 8. TRACE từng bước

Ta theo dõi (a) vòng lặp tự viết, in từng ô; (b) pipeline `isna → fillna → kiểm tra`.

In [18]:
def trace_loop(products):
    df = products.copy()
    new_quantity = []
    for step, (label, name, q) in enumerate(zip(df.index, df["name"], df["quantity"]), start=1):
        missing = pd.isna(q)
        value = 0 if missing else q
        print(f"bước {step}: nhãn={label}, name={name!r}, quantity={q!r}, thiếu={missing} -> ghi {value!r}")
        new_quantity.append(value)
        print(f"          new_quantity = {new_quantity}")
    df["quantity"] = pd.Series(new_quantity, index=df.index, dtype="float64")
    print("\ngán vào cột ->", df["quantity"].tolist())
    return df


_ = trace_loop(products)

bước 1: nhãn=0, name='Wristwatch', quantity=nan, thiếu=True -> ghi 0
          new_quantity = [0]
bước 2: nhãn=1, name='WirelessEarbuds', quantity=nan, thiếu=True -> ghi 0
          new_quantity = [0, 0]
bước 3: nhãn=2, name='GolfClubs', quantity=779.0, thiếu=False -> ghi 779.0
          new_quantity = [0, 0, 779.0]
bước 4: nhãn=3, name='Printer', quantity=849.0, thiếu=False -> ghi 849.0
          new_quantity = [0, 0, 779.0, 849.0]

gán vào cột -> [0.0, 0.0, 779.0, 849.0]


In [19]:
def trace_fillna(products):
    df = products.copy()
    print("BƯỚC 0 - bảng gốc", df.shape)
    print(df)
    print("ô thiếu ở quantity:", df["quantity"].isna().sum())

    filled = df["quantity"].fillna(0)
    print("\nBƯỚC 1 - fillna(0) trả Series mới:", filled.tolist())
    print("cột trong bảng lúc này vẫn:", df["quantity"].tolist())

    df["quantity"] = filled
    print("\nBƯỚC 2 - gán lại vào cột:")
    print(df)
    print("ô thiếu còn lại:", df["quantity"].isna().sum(), "| dtype:", df["quantity"].dtype)

    same = df.equals(products.fillna({"quantity": 0}))
    print("\ngiống fillna theo từ điển?", same)
    return df


_ = trace_fillna(products)

BƯỚC 0 - bảng gốc (4, 3)
              name  quantity  price
0       Wristwatch       NaN    135
1  WirelessEarbuds       NaN    821
2        GolfClubs     779.0   9319
3          Printer     849.0   3051
ô thiếu ở quantity: 2

BƯỚC 1 - fillna(0) trả Series mới: [0.0, 0.0, 779.0, 849.0]
cột trong bảng lúc này vẫn: [nan, nan, 779.0, 849.0]

BƯỚC 2 - gán lại vào cột:
              name  quantity  price
0       Wristwatch       0.0    135
1  WirelessEarbuds       0.0    821
2        GolfClubs     779.0   9319
3          Printer     849.0   3051
ô thiếu còn lại: 0 | dtype: float64

giống fillna theo từ điển? True


Bảng tóm tắt:

| bước | đối tượng | giá trị |
|---|---|---|
| 0 | `quantity` | `[nan, nan, 779.0, 849.0]` |
| 1 | `quantity.fillna(0)` | Series `[0.0, 0.0, 779.0, 849.0]` (bảng chưa đổi) |
| 2 | gán vào cột | bảng không còn ô thiếu; `name`, `price` không đổi |

## 9. Code nộp LeetCode

Bài pandas không có `class Solution`: chỉ cần định nghĩa hàm đúng tên và chữ ký (đối chiếu lại với đề gốc).

In [20]:
import pandas as pd

def fillMissingValues(products: pd.DataFrame) -> pd.DataFrame:
    products["quantity"] = products["quantity"].fillna(0)
    return products


demo = products.copy()           # dùng bản sao để các cell sau còn dữ liệu gốc
print(fillMissingValues(demo))

              name  quantity  price
0       Wristwatch       0.0    135
1  WirelessEarbuds       0.0    821
2        GolfClubs     779.0   9319
3          Printer     849.0   3051


## 10. Vì sao thuật toán đúng

- **Đúng ô cần lấp:** `fillna` chỉ đổi những ô mà `isna()` là `True`; ô có giá trị (kể cả số `0` sẵn có) không bị đổi.
- **Đúng giá trị:** mọi ô thiếu trở thành `0`, đúng yêu cầu.
- **Đúng vị trí:** phép gán `df["quantity"] = Series` ghép theo nhãn index; Series được tính từ chính cột đó nên giá trị mới nằm đúng hàng cũ.
- **Cột khác không đổi:** ta chỉ gán vào `quantity`; `name`, `price` giữ nguyên giá trị, kiểu và thứ tự.
- **Tính chất kiểm được:** sau khi lấp, `isna().sum() == 0` ở cột `quantity`; ở các hàng không thiếu giá trị giống hệt cũ; số hàng không đổi.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng, `c` là số cột.

- **Time:** `O(m)`: mỗi ô của cột `quantity` được xem đúng một lần. `fillna` chạy bằng mã C nên hằng số nhỏ; vòng lặp Python có hằng số lớn hơn nhưng cùng bậc.
- **Space:** `O(m)` cho cột mới / mặt nạ; `O(m·c)` nếu tạo bảng mới (`fillna` trên bảng, `copy`).
- Không thể nhanh hơn `O(m)` vì phải kiểm tra mọi ô ít nhất một lần.

## 12. Bộ test

Ta đưa các cách vào từ điển `impls`; mỗi lần gọi truyền **bản sao** của bảng để các hàm sửa tại chỗ không làm hỏng test sau.

In [21]:
impls = {
    "loop": fill_loop,
    "fillna": fill_fillna,
    "dict": fill_dict,
    "where": fill_where,
    "mask": fill_mask,
    "loc": fill_loc,
    "submit": fillMissingValues,
}

expected_example = pd.DataFrame({
    "name": ["Wristwatch", "WirelessEarbuds", "GolfClubs", "Printer"],
    "quantity": [0.0, 0.0, 779.0, 849.0],
    "price": [135, 821, 9319, 3051],
})

for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(products.copy()), expected_example)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [22]:
def make(quantities):
    n = len(quantities)
    return pd.DataFrame({
        "name": pd.Series([f"p{i}" for i in range(n)], dtype="str"),
        "quantity": pd.Series(quantities, dtype="float64"),
        "price": pd.Series([10] * n, dtype="int64"),
    })


def check_all(df, expected, label):
    for key, fn in impls.items():
        got = fn(df.copy())["quantity"].tolist()
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all(make([np.nan]), [0.0], "một hàng thiếu")
check_all(make([5.0]), [5.0], "một hàng có giá trị")
check_all(make([np.nan, np.nan, np.nan]), [0.0, 0.0, 0.0], "thiếu toàn bộ")
check_all(make([1.0, 2.0, 3.0]), [1.0, 2.0, 3.0], "không thiếu gì")
check_all(make([0.0, np.nan, 0.0]), [0.0, 0.0, 0.0], "số 0 sẵn có không bị ảnh hưởng")
check_all(make([np.nan, 9.0, np.nan]), [0.0, 9.0, 0.0], "thiếu xen kẽ")

PASS một hàng thiếu -> [0.0]
PASS một hàng có giá trị -> [5.0]
PASS thiếu toàn bộ -> [0.0, 0.0, 0.0]
PASS không thiếu gì -> [1.0, 2.0, 3.0]
PASS số 0 sẵn có không bị ảnh hưởng -> [0.0, 0.0, 0.0]
PASS thiếu xen kẽ -> [0.0, 9.0, 0.0]


### 12.2. Bảng rỗng, thiếu ở cột khác, đầu vào, và phản ví dụ

In [23]:
# Bảng rỗng
empty = make([])
for key, fn in impls.items():
    out = fn(empty.copy())
    assert len(out) == 0 and list(out.columns) == ["name", "quantity", "price"], key
print("PASS bảng rỗng")

# Thiếu ở cột khác (price) thì KHÔNG được lấp
df = pd.DataFrame({"name": ["a", "b"], "quantity": [np.nan, 1.0], "price": [np.nan, 2.0]})
for key, fn in impls.items():
    out = fn(df.copy())
    assert out["quantity"].tolist() == [0.0, 1.0], key
    assert out["price"].isna().tolist() == [True, False], key
print("PASS chỉ lấp cột quantity; price vẫn còn ô thiếu")

# Các cách "thuần" không sửa đầu vào
before = products.copy()
for key in ["loop", "dict", "where", "mask"]:
    impls[key](products)
pd.testing.assert_frame_equal(products, before)
print("PASS các cách thuần không sửa đầu vào")

# Tính chất kết quả
out = fillMissingValues(products.copy())
assert out["quantity"].isna().sum() == 0 and len(out) == len(products)
assert out["name"].tolist() == products["name"].tolist()
keep = products["quantity"].notna()
assert out.loc[keep, "quantity"].tolist() == products.loc[keep, "quantity"].tolist()
print("PASS tính chất kết quả")

# Phản ví dụ: fillna(0) trên cả bảng làm hỏng cột chữ có ô thiếu
bad = pd.DataFrame({"name": ["a", None], "quantity": [np.nan, 1.0]})
wrong = bad.fillna(0)
assert wrong["name"].tolist()[1] == 0                  # tên bị thay bằng số 0!
print("PASS phản ví dụ: fillna(0) cả bảng nhét số 0 vào cột name ->", wrong["name"].tolist())

PASS bảng rỗng
PASS chỉ lấp cột quantity; price vẫn còn ô thiếu
PASS các cách thuần không sửa đầu vào
PASS tính chất kết quả
PASS phản ví dụ: fillna(0) cả bảng nhét số 0 vào cột name -> ['a', 0]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [24]:
import random

random.seed(2887)
for trial in range(200):
    n = random.randint(0, 20)
    p_missing = random.choice([0.0, 0.3, 0.7, 1.0])
    qs = [np.nan if random.random() < p_missing else float(random.randint(0, 50)) for _ in range(n)]
    df = make(qs)
    ref = df.assign(quantity=pd.Series([0.0 if np.isnan(v) else v for v in qs], dtype="float64"))
    for key, fn in impls.items():
        pd.testing.assert_frame_equal(fn(df.copy()), ref)
print("PASS 200 bảng ngẫu nhiên - mọi cách khớp nhau")

PASS 200 bảng ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `fillna(0)` trên cả bảng — làm hỏng cột khác

In [25]:
df = pd.DataFrame({"name": ["Watch", None], "quantity": [np.nan, 4.0]})
sai = df.fillna(0)
print("sai  :", sai["name"].tolist())            # ['Watch', 0]  <- số 0 lẫn trong cột tên
dung = df.fillna({"quantity": 0})
print("đúng :", dung["name"].tolist(), "|", dung["quantity"].tolist())

sai  : ['Watch', 0]
đúng : ['Watch', nan] | [0.0, 4.0]


**Vì sao sai:** `df.fillna(0)` điền `0` vào **mọi cột** có ô thiếu, kể cả cột chữ, và biến cột `name` thành hỗn hợp chữ + số mà không báo lỗi. Đề chỉ yêu cầu cột `quantity`: dùng `df["quantity"].fillna(0)` hoặc `df.fillna({"quantity": 0})`.

### Lỗi 2: Gọi `fillna` nhưng quên gán lại

In [26]:
def sai_quen_gan(products):
    products["quantity"].fillna(0)            # tạo Series mới rồi vứt đi
    return products


print("sai  :", sai_quen_gan(products.copy())["quantity"].tolist())
print("đúng :", fillMissingValues(products.copy())["quantity"].tolist())

sai  : [nan, nan, 779.0, 849.0]
đúng : [0.0, 0.0, 779.0, 849.0]


**Vì sao sai:** `fillna` mặc định **trả về** kết quả và không sửa dữ liệu cũ. Phải gán lại `products["quantity"] = ...` hoặc dùng `loc`.

### Lỗi 3: `inplace=True` trên một cột (chained assignment)

In [27]:
import warnings

tmp = products.copy()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    tmp["quantity"].fillna(0, inplace=True)          # sửa tại chỗ... trên một bản tạm
print("bảng sau khi gọi:", tmp["quantity"].tolist())     # vẫn còn nan!
print("pandas cảnh báo   :", [w.category.__name__ for w in caught])

bảng sau khi gọi: [nan, nan, 779.0, 849.0]
pandas cảnh báo   : ['ChainedAssignmentError']


**Vì sao sai:** `tmp["quantity"]` tạo ra một **đối tượng tạm**; `inplace=True` sửa đối tượng tạm đó chứ **không sửa bảng**. Trước đây (pandas cũ) đôi khi vô tình chạy được; từ pandas 3 thì **không bao giờ** có hiệu lực và còn kèm cảnh báo `ChainedAssignmentError` (ở trên ta bắt cảnh báo bằng `catch_warnings` để notebook không in cảnh báo). Cách đúng: gán lại `df["quantity"] = df["quantity"].fillna(0)`.

### Lỗi 4: Dùng `fillna(method="ffill")` kiểu cũ

In [28]:
s = pd.Series([1.0, np.nan, np.nan, 4.0])
try:
    s.fillna(method="ffill")
except TypeError as e:
    print("TypeError:", e)
print("đúng: ffill() ->", s.ffill().tolist())        # lấp bằng giá trị phía trước
print("đúng: bfill() ->", s.bfill().tolist())        # lấp bằng giá trị phía sau

TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'
đúng: ffill() -> [1.0, 1.0, 1.0, 4.0]
đúng: bfill() -> [1.0, 4.0, 4.0, 4.0]


**Vì sao sai:** tham số `method=` của `fillna` đã bị **gỡ** ở pandas 3. Dùng trực tiếp `ffill()` (lấp bằng giá trị gần nhất phía trước) hoặc `bfill()` (phía sau). Bài này chỉ cần điền hằng số `0`, nhưng biến thể thường gặp là điền theo giá trị lân cận.

### Lỗi 5: Mong cột tự trở về số nguyên

In [29]:
res = fillMissingValues(products.copy())
print("kiểu sau fillna:", res["quantity"].dtype)               # float64
print("giá trị        :", res["quantity"].tolist())             # [0.0, 0.0, 779.0, 849.0]
print("nếu cần int    :", res["quantity"].astype(int).tolist())

kiểu sau fillna: float64
giá trị        : [0.0, 0.0, 779.0, 849.0]
nếu cần int    : [0, 0, 779, 849]


**Lưu ý:** `fillna` **không đổi kiểu** cột. Vì cột đã là `float64` do `NaN`, kết quả vẫn là `0.0`, `779.0`... Nếu đề gốc yêu cầu kiểu nguyên thì thêm `.astype(int)`; còn nếu so sánh giá trị, `0.0 == 0` nên không ảnh hưởng. Hãy đối chiếu đề.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "điền / thay giá trị thiếu / fill missing" → nghĩ tới **`fillna`**.
- Công thức nhớ: **`df["col"] = df["col"].fillna(giá_trị)`**; nhiều cột: **`df.fillna({"a": 0, "b": "N/A"})`**.
- **Không** gọi `df.fillna(0)` trên cả bảng nếu có cột chữ.
- Điền theo **thống kê**: `s.fillna(s.mean())`, `s.fillna(s.median())`; theo **lân cận**: `s.ffill()`, `s.bfill()`.
- Điền theo **điều kiện** khác "thiếu": `s.where(...)`, `s.mask(...)`, `df.loc[mặt_nạ, "col"] = ...`.
- Đếm ô thiếu trước/sau: `df.isna().sum()`; đừng quên kiểm tra `dtype`.

## 15. Biến thể của bài

1. Lấp bằng **trung bình** của cột: `df["quantity"].fillna(df["quantity"].mean())`.
2. Lấp bằng **giá trị phía trước**: `df["quantity"].ffill()`; phía sau: `bfill()`.
3. Lấp **mỗi cột một giá trị**: `df.fillna({"quantity": 0, "price": -1, "name": "Unknown"})`.
4. Lấp rồi **đổi sang số nguyên**: `df["quantity"].fillna(0).astype(int)`.
5. Lấp theo **nhóm** (ví dụ trung bình của từng loại): `df["quantity"].fillna(df.groupby("type")["quantity"].transform("mean"))`.
6. Chỉ **thêm cột cờ** đánh dấu ô từng thiếu: `df["was_missing"] = df["quantity"].isna()`.

Ta thử vài biến thể:

In [30]:
v = products.copy()
print("1. trung bình:", v["quantity"].fillna(v["quantity"].mean()).tolist())    # mean bỏ qua NaN: (779+849)/2 = 814
print("2. ffill     :", v["quantity"].ffill().tolist())
print("2. bfill     :", v["quantity"].bfill().tolist())
print("4. int       :", v["quantity"].fillna(0).astype(int).tolist())
w = pd.DataFrame({"type": ["a", "a", "b", "b"], "quantity": [1.0, np.nan, 10.0, np.nan]})
print("5. theo nhóm :", w["quantity"].fillna(w.groupby("type")["quantity"].transform("mean")).tolist())

1. trung bình: [814.0, 814.0, 779.0, 849.0]
2. ffill     : [nan, nan, 779.0, 849.0]
2. bfill     : [779.0, 779.0, 779.0, 849.0]
4. int       : [0, 0, 779, 849]
5. theo nhóm : [1.0, 1.0, 10.0, 10.0]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp + `pd.isna` | tự thay từng ô | `O(m)` (Python, chậm) | `O(m)` | hiểu bản chất |
| 2. `df["col"] = df["col"].fillna(0)` | lấp cột + gán | `O(m)` | `O(m)` | **cách nộp bài** |
| 3. `df.fillna({"col": 0})` | lấp theo từ điển, bảng mới | `O(m)` | `O(m)` đến `O(m·c)` | nhiều cột, mỗi cột một giá trị |
| 4. `where` / `mask` | giữ/thay theo điều kiện | `O(m)` | `O(m)` | điều kiện thay thế phức tạp hơn "thiếu" |
| 5. `loc[mask, "col"] = 0` | gán có điều kiện tại chỗ | `O(m)` | `O(m)` | gán cho ô thỏa điều kiện bất kỳ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `pd.Series([1.0, None]).isna().tolist()`, `pd.Series([1.0, None]).fillna(9).tolist()`, `products["quantity"].dtype`, `products["quantity"].fillna(0).dtype`, `products["quantity"].isna().sum()`.

**Bài 2 (tính tay).** Với `q = [None, 4, None, 6]`: viết kết quả `fillna(0)`, `ffill()`, `bfill()` và `fillna(q.mean())` (trung bình bỏ qua ô thiếu).

**Bài 3 (điền TODO).** Hoàn thành `fill_todo(df)` bằng `fillna`, không sửa bảng đầu vào.

In [31]:
def fill_todo(df):
    # TODO 1: out = df.copy()
    # TODO 2: out["quantity"] = out["quantity"].fillna(0)
    # TODO 3: return out
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `fill_loop2(products)` bằng vòng lặp thuần Python (Cách 1) mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `fill_with_mean(df)`: lấp `quantity` thiếu bằng **trung bình** các ô có giá trị, làm tròn xuống số nguyên (`int`), trả cột kiểu `float64`. (b) Viết `fill_columns(df, rules)` nhận từ điển `{cột: giá_trị}` và chỉ lấp những cột đó, không sửa bảng đầu vào.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[False, True]`, `[1.0, 9.0]`, `float64`, `float64`, `2`. Bài 2: `ffill` lấy giá trị bên trên, `bfill` lấy giá trị bên dưới.
- **Mức 2 (vừa):** Bài 2: `fillna(0)` -> `[0, 4, 0, 6]`; `ffill()` -> `[NaN, 4, 4, 6]` (ô đầu không có gì phía trước nên vẫn thiếu); `bfill()` -> `[4, 4, 6, 6]`; mean = `5` -> `[5, 4, 5, 6]`. Bài 3: ba dòng đã ghi trong comment. Bài 4: duyệt từng `q`, nếu `pd.isna(q)` thì thêm `0`, không thì thêm `q`.
- **Mức 3 (sâu):** Bài 5(a): `m = int(df["quantity"].mean())` rồi `df.assign(quantity=df["quantity"].fillna(m))`. Bài 5(b): `df.fillna(rules)` (trả bảng mới); hoặc `df.fillna({c: v for c, v in rules.items() if c in df.columns})` để an toàn.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert pd.Series([1.0, None]).isna().tolist() == [False, True]
assert pd.Series([1.0, None]).fillna(9).tolist() == [1.0, 9.0]
assert str(products["quantity"].dtype) == "float64"
assert str(products["quantity"].fillna(0).dtype) == "float64"
assert products["quantity"].isna().sum() == 2

# Bài 2
q2 = pd.Series([None, 4, None, 6], dtype="float64")
assert q2.fillna(0).tolist() == [0.0, 4.0, 0.0, 6.0]
assert q2.ffill().isna().tolist() == [True, False, False, False]
assert q2.ffill().tolist()[1:] == [4.0, 4.0, 6.0]
assert q2.bfill().tolist() == [4.0, 4.0, 6.0, 6.0]
assert q2.fillna(q2.mean()).tolist() == [5.0, 4.0, 5.0, 6.0]

# Bài 3
def fill_todo_ans(df):
    out = df.copy()
    out["quantity"] = out["quantity"].fillna(0)
    return out

before = products.copy()
pd.testing.assert_frame_equal(fill_todo_ans(products), expected_example)
pd.testing.assert_frame_equal(products, before)      # bảng đầu vào không đổi

# Bài 4
def fill_loop2(products):
    df = products.copy()
    new_quantity = []
    for q in df["quantity"]:
        if pd.isna(q):
            new_quantity.append(0)
        else:
            new_quantity.append(q)
    df["quantity"] = pd.Series(new_quantity, index=df.index, dtype="float64")
    return df

pd.testing.assert_frame_equal(fill_loop2(products), expected_example)

# Bài 5(a)
def fill_with_mean(df):
    m = int(df["quantity"].mean())
    return df.assign(quantity=df["quantity"].fillna(m))

r = fill_with_mean(products)
assert r["quantity"].tolist() == [814.0, 814.0, 779.0, 849.0]

# Bài 5(b)
def fill_columns(df, rules):
    return df.fillna({c: v for c, v in rules.items() if c in df.columns})

t = pd.DataFrame({"name": ["a", None], "quantity": [np.nan, 1.0]})
r = fill_columns(t, {"quantity": 0, "khong_co": 5})
assert r["quantity"].tolist() == [0.0, 1.0] and r["name"].isna().tolist() == [False, True]
assert t["quantity"].isna().sum() == 1        # đầu vào không đổi
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (điền giá trị thiếu ở một cột)

```python
import pandas as pd

def fillMissingValues(products: pd.DataFrame) -> pd.DataFrame:
    products["quantity"] = products["quantity"].fillna(0)
    return products
```

**Checklist**

- [ ] Có **phép gán** `products["quantity"] = ...` (hoặc dùng `fillna({...})` / `loc` rồi `return`)?
- [ ] Chỉ lấp cột `quantity` (không `fillna(0)` trên cả bảng khi còn cột chữ)?
- [ ] Không dùng `inplace=True` trên một cột (chained assignment không có tác dụng ở pandas 3)?
- [ ] Không dùng `fillna(method=...)` (đã gỡ); cần lấp theo lân cận thì `ffill()` / `bfill()`?
- [ ] Biết cột vẫn `float64` sau `fillna`; thêm `astype(int)` chỉ khi đề yêu cầu?
- [ ] **Trả về** DataFrame, các cột `name`, `price` và thứ tự hàng không đổi?
- [ ] Đã thử bảng rỗng, không có ô thiếu, toàn thiếu?
- [ ] Skeleton đúng `def fillMissingValues(products: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?